# Query a Fabric Data Agent through MCP

Connect directly to the published Fabric Data Agent MCP endpoint, inspect the tools it exposes, and call one of those tools with a natural-language question. The endpoint uses the signed-in Azure Developer CLI identity and requires access to the Fabric workspace.

## Step 1: Load configuration and create an MCP session helper

Load the Fabric tenant and MCP endpoint written to the project `.env` file during provisioning. The HTTP client follows redirects because Fabric routes MCP requests through its service endpoint.

In [2]:
import os
from contextlib import asynccontextmanager

import httpx
from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_DATA_AGENT_MCP_URL = os.environ["FABRIC_DATA_AGENT_MCP_URL"]
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"


@asynccontextmanager
async def open_data_agent_session():
    credential = AzureDeveloperCliCredential(tenant_id=FABRIC_TENANT_ID)
    token = credential.get_token(FABRIC_SCOPE)

    async with httpx.AsyncClient(
        headers={"Authorization": f"Bearer {token.token}"},
        follow_redirects=True,
        timeout=httpx.Timeout(30, read=300),
    ) as http_client:
        async with streamable_http_client(FABRIC_DATA_AGENT_MCP_URL, http_client=http_client,) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

print("Fabric Data Agent configuration loaded")

Fabric Data Agent configuration loaded


## Step 2: List the available tools

Initialize an authenticated MCP session and inspect every tool exposed by the published data agent, including each tool's input schema.

In [3]:
from rich.console import Console

console = Console()

async with open_data_agent_session() as session:
    tools_response = await session.list_tools()

data_agent_tools = tools_response.tools

for tool in data_agent_tools:
    console.print(f"[bold cyan]Tool:[/] {tool.name}")
    if tool.description:
        console.print(f"[bold]Description:[/] {tool.description}")
    console.print("[bold]Input schema:[/]")
    console.print_json(data=tool.inputSchema)
    console.print()

Tool: DataAgent_ContosoDIYDataAgent

Description: Contoso DIY product operations and review intelligence data agent

Input schema:

{
  "type": "object",
  "properties": {
    "userQuestion": {
      "type": "string"
    }
  },
  "required": [
    "userQuestion"
  ]
}

## Step 3: Call the Fabric Data Agent tool

Use the first tool discovered above and derive its question argument from the advertised input schema. Change `question` to explore other product and inventory questions.

In [4]:
from rich.markdown import Markdown

question = "Which products currently have the lowest inventory quantities?"
tool = data_agent_tools[0]
properties = tool.inputSchema.get("properties", {})

question_argument = next(iter(properties))
async with open_data_agent_session() as session:
    result = await session.call_tool(
        tool.name,
        {question_argument: question},
    )

answers = [block.text for block in result.content if block.type == "text"]
console.print(Markdown("\n\n".join(answers)))

Here are the products with the lowest available inventory quantities (many with zero or only one item available),  
across various Contoso DIY stores:                                                                                 

Products with 0 Available Quantity                                                                                 

 • Fiberglass Pipe Insulation (SKU: PLPI000009)                                                                    
 • Adjustable Wrench 10-inch (SKU: HTWR000003)                                                                     
 • Deadbolt Lock Keyed (SKU: HWLO000031)                                                                           
 • Butt Hinge 3-1/2 inch (SKU: HWHI000021)                                                                         
 • Pressure Treated 2x4x8 (SKU: LBDL000001)                                                                        
 • Rolling Tool Chest 26-inch (SKU: SOTLB001)                                                                      
 • Retractable Cord Reel (SKU: ELEC000019)                                                                         
 • Fill Valve Assembly (SKU: PLTLT002)                                                                             
 • Cedar Board 1x8x10 (SKU: LBTRM003)                                                                              
 • Kitchen Sink Faucet (SKU: PLFCT001)                                                                             
 • Shower Faucet Trim Kit (SKU: PLFCT004)                                                                          
 • Crown Molding Pine 8ft (SKU: LBMLD001)                                                                          
 • Metric Wrench Set (SKU: HTWR000002)                                                                             
 • Hex Bolt Kit Grade 5 (SKU: HWBO000011)                                                                          
 • Drywall Sander (SKU: PTSA000015)                                                                                
 • Gloss Polyurethane (SKU: PFPU000037)                                                                            
 • Combination Wrench Set SAE (SKU: HTWR000001)                                                                    
 • Solar Path Light Set (SKU: GOOTD001)                                                                            
 • Douglas Fir 2x6x10 (SKU: LBDL000002)                                                                            
 • Ball Valve 1/2-inch (SKU: PLVA000001)                                                                           
 • Toilet Seat Standard (SKU: PLTLT005)                                                                            
 • Chemical Storage Cabinet (SKU: SOCA000019)                                                                      
 • Carriage Bolt Set (SKU: HWBO000012)                                                                             
 • Countertop Support Bracket (SKU: HWBK000045)                                                                    
 • File Belt Sander (SKU: PTBS000043)                                                                              
 • Standard Drywall 4x8x1/2 (SKU: LBDW000006)                                                                      
 • Ceiling Box with Bracket (SKU: ELJB000013)                                                                      
 • Wall Mount Cabinet (SKU: SOCA000017)                                                                            
 • Track Lighting Kit (SKU: ELLF000024)                                                                            
 • Rust Prevention Spray (SKU: PFSP000029)                                                                         
 • Push-Fit Connectors (SKU: PLPPF004)                                                                             
 • Rubber Washer Set (SKU: HWWA000019)                